# 01 — Data Exploration
**NewsBot Intelligence System 2.0 | ITAI 2373**

Load the BBC News dataset, explore its structure, and prepare `df_final` for downstream modules.

In [ ]:
%pip install kaggle pandas matplotlib seaborn wordcloud -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import os, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'#0f1117','axes.facecolor':'#1a1d2e',
    'axes.edgecolor':'#2d3561','axes.labelcolor':'#e8d5a3',
    'xtick.color':'#a0a8c0','ytick.color':'#a0a8c0',
    'text.color':'#e8d5a3','grid.color':'#2d3561',
    'grid.alpha':0.5,'figure.dpi':120
})
PALETTE = {'tech':'#4fc3f7','business':'#81c784','politics':'#e57373',
           'sport':'#ffb74d','entertainment':'#ce93d8'}
print('Setup complete.')


## 1.1 — Load Dataset

In [ ]:
# ── Kaggle download ───────────────────────────────────────────────
import os
if not os.path.exists('/content/bbc-news-data.csv'):
    from google.colab import files
    print('Upload kaggle.json')
    files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d shivamkushwaha/bbc-full-text-document-classification --unzip -p /content/
    print('Dataset downloaded.')

# ── Load ──────────────────────────────────────────────────────────
for fname in ['bbc-news-data.csv','bbc_news.csv','bbc.csv']:
    if os.path.exists(f'/content/{fname}'):
        df = pd.read_csv(f'/content/{fname}')
        break

df.columns = [c.lower().strip() for c in df.columns]
if 'text' not in df.columns and 'article' in df.columns:
    df.rename(columns={'article':'text'}, inplace=True)
if 'category' not in df.columns and 'labels' in df.columns:
    df.rename(columns={'labels':'category'}, inplace=True)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)


## 1.2 — Basic Statistics

In [ ]:
print('='*50)
print('  BBC NEWS DATASET OVERVIEW')
print('='*50)
print(f'  Total articles : {len(df):,}')
print(f'  Categories     : {df.category.nunique()}')
print(f'  Null values    : {df.isnull().sum().sum()}')
print(f'  Duplicates     : {df.text.duplicated().sum()}')
print()
print('Category distribution:')
print(df.category.value_counts())


In [ ]:
# Category bar chart
cat_counts = df.category.value_counts()
colors = [PALETTE.get(c,'#ffffff') for c in cat_counts.index]

fig, ax = plt.subplots(figsize=(10,5))
bars = ax.bar(cat_counts.index, cat_counts.values, color=colors, alpha=0.9, width=0.6)
for bar,cnt in zip(bars, cat_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            str(cnt), ha='center', fontsize=11, fontweight='bold')
ax.set_title('BBC News — Category Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Category'); ax.set_ylabel('Article Count')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('category_distribution.png',bbox_inches='tight',dpi=150,facecolor='#0f1117')
plt.show()


## 1.3 — Text Length Analysis

In [ ]:
df['text_length'] = df.text.str.len()
df['word_count']  = df.text.str.split().str.len()

print('Text length statistics:')
print(df.groupby('category')[['text_length','word_count']].mean().round(0))

fig, axes = plt.subplots(1,2,figsize=(14,5))
for i,(col,label) in enumerate([('text_length','Character Count'),('word_count','Word Count')]):
    ax = axes[i]
    for cat in df.category.unique():
        subset = df[df.category==cat][col]
        ax.hist(subset, bins=40, alpha=0.6, color=PALETTE.get(cat,'#fff'), label=cat, density=True)
    ax.set_title(f'{label} Distribution', fontsize=12, fontweight='bold')
    ax.set_xlabel(label); ax.set_ylabel('Density')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('text_length_distribution.png',bbox_inches='tight',dpi=150,facecolor='#0f1117')
plt.show()


## 1.4 — Word Clouds per Category

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
categories = sorted(df.category.unique())

for ax, cat in zip(axes, categories):
    text = ' '.join(df[df.category==cat].text.tolist())
    wc = WordCloud(width=400, height=300, background_color='#1a1d2e',
                   colormap='Blues', max_words=80,
                   stopwords={'the','and','to','of','a','in','is','it','that','for'}).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(cat.upper(), fontsize=11, fontweight='bold', color=PALETTE.get(cat,'#fff'))
    ax.axis('off')

plt.suptitle('Word Clouds by Category', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('wordclouds.png',bbox_inches='tight',dpi=150,facecolor='#0f1117')
plt.show()


## 1.5 — Sample Articles

In [ ]:
print('Sample article from each category:\n')
for cat in sorted(df.category.unique()):
    sample = df[df.category==cat].sample(1).iloc[0]
    print(f'[{cat.upper()}]')
    print(sample.text[:300]+'...')
    print()


## 1.6 — Save df_final

In [ ]:
# df_final is the base DataFrame passed to all subsequent notebooks
df_final = df.copy()
df_final = df_final.dropna(subset=['text','category']).reset_index(drop=True)
df_final['doc_id'] = df_final.index

print(f'df_final ready: {df_final.shape}')
print(f'Columns: {list(df_final.columns)}')
df_final.head(3)
